# Model Context Protocol (MCP)

Model Context Protocol (MCP) is an open protocol that standardizes how applications provide tools and context to LLMs. LangChain agents can use tools defined on MCP servers using the langchain-mcp-adapters library.

>>>>>> Basically, <b>it is tool server</b>, but with standard 

>>>>>> think of old phone charger, there're various type

>>>>>> but <b>with MCP, it is similar to what USB-C did to all chargers. United it all</b>

It is a protocol (rule for tool to follow, if not follow it's just tool not considered as MCP tool):

4 rules summarized by Gemini 



<b>Rule 1: The "Self-Discovery" Rule (Naming & Metadata) (Discovery)</b>
In a normal script, you know what a function does because you wrote it. In MCP, the server must tell the AI what is available.

The Rule: Your function must have a Unique Name (no spaces, usually snake_case) and a Clear Description.

<b>Why?</b> The AI uses the description to decide when to use the tool. If your description is "Calculates thing," the AI will never call it. If it’s "Calculates the annual interest rate for a banking customer based on their credit score," the AI knows exactly when to trigger it.

==========================================================================

<b>Rule 2: The "Strict Input" Rule (JSON Schema) (Schema)</b>
You cannot just pass a "variable" like you do in Python. You must define a Schema.

The Rule: Every input variable must have a defined Type (string, integer, boolean, etc.) and a Description.

sender (AI) will send exactly this format

In [ ]:
{
    "jsonrpc": "2.0",
    "id": "request-123",
    "method": "tools/call",
    "params": {
        "name": "calculate_ltv",
        "arguments": {
        "argument1": 85000,
        "argument2": 100000
    }
  }
}

<b>jsonrpc</b>: Always "2.0". This tells your server which version of the "language" is being spoken.

<b>id</b>: A unique string or number. Your server must include this same ID in its reply so the AI knows which answer belongs to which question.

<b>method</b>: Always "tools/call" for tool execution.

<b>params.name</b>: The exact string name of your function (the one you put in the @mcp.tool() decorator).

<b>params.arguments</b>: This is the "payload." It is a dictionary where the keys match your function's argument names.


<b>Why?</b> This creates the "form" the AI fills out. The AI reads the schema and says, "Okay, I need to provide a customer_id which must be an integer."

==========================================================================

<b>Rule 3: The "Standard Envelope" Rule (JSON-RPC) (Protocol)</b>
This is the technical "algorithm" of how messages move.

The Rule: All communication must be wrapped in a JSON-RPC 2.0 envelope.

In [2]:
{
  "jsonrpc": "2.0",
  "id": "request-123",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "The calculated LTV is 85.00%"
      }
    ]
  }
}

{'jsonrpc': '2.0',
 'id': 'request-123',
 'result': {'content': [{'type': 'text',
    'text': 'The calculated LTV is 85.00%'}]}}

<b>Why?</b> It ensures that the "request" and "response" are never mixed up. Every message has an ID. When the server replies, it includes that same ID so the AI knows which question the answer belongs to.

==========================================================================

<b>Rule 4: The "Content Block" Rule (Output Format) (Output)</b>
A normal Python function might return a simple string or a list. An MCP tool must return a specific Content Object.

The Rule: You must wrap your result in a "content" array, usually as a text block: {"content": [{"type": "text", "text": "YOUR_RESULT_HERE"}]}.

<b>Why?</b> MCP supports more than just text. By following this rule, your tool could technically return images, file links, or even audio, and the AI will know how to handle each type.

pip install langchain-mcp-adapters

<b>langchain-mcp-adapters</b> enables agents to use tools defined across one or more MCP servers.

MultiServerMCPClient is <b>stateless by default</b>. 

1. Each tool invocation creates a fresh MCP ClientSession
2. executes the tool
3. then cleans up. 

See the stateful sessions section for more details.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient  ## Import here
from langchain.agents import create_agent


client = MultiServerMCPClient(  ## define
    {
        "math": {
            "transport": "stdio",  # Local subprocess communication
            "command": "python",
            # Absolute path to your math_server.py file
            "args": ["/path/to/math_server.py"],
        },
        "weather": {
            "transport": "http",  # HTTP-based remote server
            # Ensure you start your weather server on port 8000
            "url": "http://localhost:8000/mcp",
        }
    }
)

tools = await client.get_tools()  ## this is not common syntax as normally await need to be in async func
# but it can use due to langchain special RECL something
agent = create_agent( 
    "claude-sonnet-4-5-20250929",
    tools 
)
## ainvoke (asynchronous invoke) not just invoke 
math_response = await agent.ainvoke( 
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)
weather_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
)

# Custom server

In [ ]:
# pip install fastmcp <Use fastmcp Lib>

In [ ]:
from fastmcp import FastMCP

# Math 
mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

# Weather
from fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

# Some Example from Gemini

## This is writing mcp tool with FastMCP

In [ ]:
from mcp.server.fastmcp import FastMCP

# 1. The "Server" (The USB Hub)
mcp = FastMCP("BankingTools")

# 2. The "Tool" (The USB Device)
@mcp.tool()
def calculate_ltv(loan_amount: float, property_value: float) -> str:
    """
    Calculate the Loan-to-Value (LTV) ratio for a mortgage application.
    Use this tool whenever a user asks about loan risks or equity.
    """
    # The 'Algorithm' behind the scenes (SDK) handles Rule 1 & 2: 
    # It converts this docstring and these types into a JSON Schema for the AI.
    
    if property_value <= 0:
        return "Error: Property value must be greater than zero."
    
    ltv = (loan_amount / property_value) * 100
    
    # Rule 4: The 'Algorithm' (SDK) automatically wraps this 
    # string into the required MCP "Content Block" format.
    return f"The calculated LTV is {ltv:.2f}%"

# No need for a print statement; the MCP server stays 'alive' 
# waiting for an AI to ask for it.

## This is to write without the mcp.tool decorator

In [ ]:
import sys
import json

def calculate_ltv(loan_amount, property_value):
    ltv = (loan_amount / property_value) * 100
    return f"{ltv:.2f}%"

# --- MANUALLY DOING WHAT THE DECORATOR DOES ---
def handle_mcp_requests():
    for line in sys.stdin: # Listen to the 'pipe'
        request = json.loads(line)
        
        # 1. Manual "Discovery" logic
        if request["method"] == "tools/list":
            response = {
                "jsonrpc": "2.0", "id": request["id"],
                "result": {"tools": [{
                    "name": "calculate_ltv",
                    "description": "Calculates loan-to-value ratio",
                    "inputSchema": { "type": "object", "properties": { ... } }
                }]}
            }
        
        # 2. Manual "Calling" logic
        elif request["method"] == "tools/call":
            args = request["params"]["arguments"]
            # Manual execution and wrapping
            data = calculate_ltv(args["loan_amount"], args["property_value"])
            response = {
                "jsonrpc": "2.0", "id": request["id"],
                "result": {"content": [{"type": "text", "text": data}]}
            }
        
        # 3. Manual Output
        sys.stdout.write(json.dumps(response) + "\n")

# This is the "Algorithm" the decorator hides from you.

# Transports

MCP supports different transport mechanisms for client-server communication.

## HTTP
The http transport (also referred to as streamable-http) uses HTTP requests for client-server communication. See the MCP HTTP transport specification for more details.

In [ ]:
client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
        }
    }
)

## Passing headers

When connecting to MCP servers over HTTP, you can include custom headers (e.g., for authentication or tracing) using the headers field in the connection configuration. This is supported for sse (deprecated by MCP spec) and streamable_http transports.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
            "headers": {  
                "Authorization": "Bearer YOUR_TOKEN",  
                "X-Custom-Header": "custom-value"
            },  
        }
    }
)
tools = await client.get_tools()
agent = create_agent("openai:gpt-4.1", tools)
response = await agent.ainvoke({"messages": "what is the weather in nyc?"})

## Authentication

The langchain-mcp-adapters library uses the official MCP SDK under the hood, which allows you to provide a custom authentication mechanism by implementing the httpx.Auth interface.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
            "auth": auth, 
        }
    }
)

## stdio

Client launches server as a subprocess and communicates via standard input/output. Best for local tools and simple setups.

In [ ]:
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": ["/path/to/math_server.py"],
        }
    }
)

## Stateful sessions

By default, MultiServerMCPClient is stateless—each tool invocation creates a fresh MCP session, executes the tool, and then cleans up.

If you need to control the lifecycle of an MCP session (for example, when working with a stateful server that maintains context across tool calls), you can create a persistent ClientSession using client.session().

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent

client = MultiServerMCPClient({...})

# Create a session explicitly
async with client.session("server_name") as session:  
    # Pass the session to load tools, resources, or prompts
    tools = await load_mcp_tools(session)  
    agent = create_agent(
        "anthropic:claude-3-7-sonnet-latest",
        tools
    )

# Core features

## Tools

Tools allow MCP servers to expose executable functions that LLMs can invoke to perform actions—such as querying databases, calling APIs, or interacting with external systems. LangChain converts MCP tools into LangChain tools, making them directly usable in any LangChain agent or workflow.

### Loading tools
Use client.get_tools() to retrieve tools from MCP servers and pass them to your agent:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient({...})
tools = await client.get_tools()  
agent = create_agent("claude-sonnet-4-5-20250929", tools)

### Structured content

MCP tools can return structured content alongside the human-readable text response. This is useful when a tool needs to return machine-parseable data (like JSON) in addition to text that gets shown to the model.

When an MCP tool returns structuredContent, the adapter wraps it in an MCPToolArtifact and returns it as the tool’s artifact. You can access this using the artifact field on the ToolMessage. You can also use interceptors to process or transform structured content automatically.

### Extracting structured content from artifact

After invoking your agent, you can access the structured content from tool messages in the response:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import ToolMessage

client = MultiServerMCPClient({...})
tools = await client.get_tools()
agent = create_agent("claude-sonnet-4-5-20250929", tools)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Get data from the server"}]}
)

# Extract structured content from tool messages
for message in result["messages"]:
    if isinstance(message, ToolMessage) and message.artifact:
        structured_content = message.artifact["structured_content"]

### Multimodal tool content

MCP tools can return multimodal content (images, text, etc.) in their responses. When an MCP server returns content with multiple parts (e.g., text and images), the adapter converts them to LangChain’s standard content blocks. You can access the standardized representation via the content_blocks property on the ToolMessage:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient({...})
tools = await client.get_tools()
agent = create_agent("claude-sonnet-4-5-20250929", tools)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Take a screenshot of the current page"}]}
)

# Access multimodal content from tool messages
for message in result["messages"]:
    if message.type == "tool":
        # Raw content in provider-native format
        print(f"Raw content: {message.content}")

        # Standardized content blocks  #
        for block in message.content_blocks:  
            if block["type"] == "text":  
                print(f"Text: {block['text']}")  
            elif block["type"] == "image":  
                print(f"Image URL: {block.get('url')}")  
                print(f"Image base64: {block.get('base64', '')[:50]}...")

This allows you to handle multimodal tool responses in a provider-agnostic way, regardless of how the underlying MCP server formats its content.

## Resources

Resources allow MCP servers to expose data—such as files, database records, or API responses—that can be read by clients. LangChain converts MCP resources into Blob objects, which provide a unified interface for handling both text and binary content.

## Loading resources
Use client.get_resources() to load resources from an MCP server:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({...})

# Load all resources from a server
blobs = await client.get_resources("server_name")  

# Or load specific resources by URI
blobs = await client.get_resources("server_name", uris=["file:///path/to/file.txt"])  

for blob in blobs:
    print(f"URI: {blob.metadata['uri']}, MIME type: {blob.mimetype}")
    print(blob.as_string())  # For text content

You can also use load_mcp_resources directly with a session for more control:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.resources import load_mcp_resources

client = MultiServerMCPClient({...})

async with client.session("server_name") as session:
    # Load all resources
    blobs = await load_mcp_resources(session)

    # Or load specific resources by URI
    blobs = await load_mcp_resources(session, uris=["file:///path/to/file.txt"])

## Prompts
Prompts allow MCP servers to expose reusable prompt templates that can be retrieved and used by clients. LangChain converts MCP prompts into messages, making them easy to integrate into chat-based workflows.
​
## Loading prompts
Use client.get_prompt() to load a prompt from an MCP server:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({...})

# Load a prompt by name
messages = await client.get_prompt("server_name", "summarize")  

# Load a prompt with arguments
messages = await client.get_prompt(  
    "server_name",  
    "code_review",  
    arguments={"language": "python", "focus": "security"}  
)  

# Use the messages in your workflow
for message in messages:
    print(f"{message.type}: {message.content}")

You can also use load_mcp_prompt directly with a session for more control:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.prompts import load_mcp_prompt

client = MultiServerMCPClient({...})

async with client.session("server_name") as session:
    # Load a prompt by name
    messages = await load_mcp_prompt(session, "summarize")

    # Load a prompt with arguments
    messages = await load_mcp_prompt(
        session,
        "code_review",
        arguments={"language": "python", "focus": "security"}
    )

# Advanced features

## Tool interceptors
<b>MCP servers run as separate processes—they can’t access LangGraph runtime information like the store, context, or agent state.</b> Interceptors bridge this gap by giving you access to this runtime context during MCP tool execution.

Interceptors also provide middleware-like control over tool calls: you can modify requests, implement retries, add headers dynamically, or short-circuit execution entirely.

<img src='MCP_accessing_runtime.jpg'>

# Accessing runtime context

When MCP tools are used within a LangChain agent (via create_agent), interceptors receive access to the ToolRuntime context. This provides access to the tool call ID, state, config, and store—enabling powerful patterns for accessing user data, persisting information, and controlling agent behavior.

Access user-specific configuration like user IDs, API keys, or permissions that are passed at invocation time:

In [ ]:
from dataclasses import dataclass
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent

@dataclass
class Context:
    user_id: str
    api_key: str

async def inject_user_context(
    request: MCPToolCallRequest,
    handler,
):
    """Inject user credentials into MCP tool calls."""
    runtime = request.runtime
    user_id = runtime.context.user_id  
    api_key = runtime.context.api_key  

    # Add user context to tool arguments
    modified_request = request.override(
        args={**request.args, "user_id": user_id}
    )
    return await handler(modified_request)

client = MultiServerMCPClient(
    {...},
    tool_interceptors=[inject_user_context],
)
tools = await client.get_tools()
agent = create_agent("gpt-4.1", tools, context_schema=Context)

# Invoke with user context
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Search my orders"}]},
    context={"user_id": "user_123", "api_key": "sk-..."}
)

Access long-term memory to retrieve user preferences or persist data across conversations:

In [ ]:
from dataclasses import dataclass
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

async def personalize_search(
    request: MCPToolCallRequest,
    handler,
):
    """Personalize MCP tool calls using stored preferences."""
    runtime = request.runtime
    user_id = runtime.context.user_id
    store = runtime.store  

    # Read user preferences from store
    prefs = store.get(("preferences",), user_id)  

    if prefs and request.name == "search":
        # Apply user's preferred language and result limit
        modified_args = {
            **request.args,
            "language": prefs.value.get("language", "en"),
            "limit": prefs.value.get("result_limit", 10),
        }
        request = request.override(args=modified_args)

    return await handler(request)

client = MultiServerMCPClient(
    {...},
    tool_interceptors=[personalize_search],
)
tools = await client.get_tools()
agent = create_agent(
    "gpt-4.1",
    tools,
    context_schema=Context,
    store=InMemoryStore()
)

Access conversation state to make decisions based on the current session:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.messages import ToolMessage

async def require_authentication(
    request: MCPToolCallRequest,
    handler,
):
    """Block sensitive MCP tools if user is not authenticated."""
    runtime = request.runtime
    state = runtime.state  
    is_authenticated = state.get("authenticated", False)  

    sensitive_tools = ["delete_file", "update_settings", "export_data"]

    if request.name in sensitive_tools and not is_authenticated:
        # Return error instead of calling tool
        return ToolMessage(
            content="Authentication required. Please log in first.",
            tool_call_id=runtime.tool_call_id,
        )

    return await handler(request)

client = MultiServerMCPClient(
    {...},
    tool_interceptors=[require_authentication],
)

Access the tool call ID to return properly formatted responses or track tool executions:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.messages import ToolMessage

async def rate_limit_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Rate limit expensive MCP tool calls."""
    runtime = request.runtime
    tool_call_id = runtime.tool_call_id  

    # Check rate limit (simplified example)
    if is_rate_limited(request.name):
        return ToolMessage(
            content="Rate limit exceeded. Please try again later.",
            tool_call_id=tool_call_id,  
        )

    result = await handler(request)

    # Log successful tool call
    log_tool_execution(tool_call_id, request.name, success=True)

    return result

client = MultiServerMCPClient(
    {...},
    tool_interceptors=[rate_limit_interceptor],
)

### State updates and commands

Interceptors can return Command objects to update agent state or control graph execution flow. This is useful for tracking task progress, switching between agents, or ending execution early.

In [ ]:
from langchain.agents import AgentState, create_agent
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.messages import ToolMessage
from langgraph.types import Command

async def handle_task_completion(
    request: MCPToolCallRequest,
    handler,
):
    """Mark task complete and hand off to summary agent."""
    result = await handler(request)

    if request.name == "submit_order":
        return Command(
            update={
                "messages": [result] if isinstance(result, ToolMessage) else [],
                "task_status": "completed",  
            },
            goto="summary_agent",  
        )

    return result

Use Command with goto="__end__" to end execution early:

In [ ]:
async def end_on_success(
    request: MCPToolCallRequest,
    handler,
):
    """End agent run when task is marked complete."""
    result = await handler(request)

    if request.name == "mark_complete":
        return Command(
            update={"messages": [result], "status": "done"},
            goto="__end__",  
        )

    return result

## Custom interceptors
Interceptors are async functions that wrap tool execution, enabling request/response modification, retry logic, and other cross-cutting concerns. They follow an “onion” pattern where the first interceptor in the list is the outermost layer.

## Basic pattern

An interceptor is an async function that receives a request and a handler. You can modify the request before calling the handler, modify the response after, or skip the handler entirely.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest

async def logging_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Log tool calls before and after execution."""
    print(f"Calling tool: {request.name} with args: {request.args}")
    result = await handler(request)
    print(f"Tool {request.name} returned: {result}")
    return result

client = MultiServerMCPClient(
    {"math": {"transport": "stdio", "command": "python", "args": ["/path/to/server.py"]}},
    tool_interceptors=[logging_interceptor],  
)

### Modifying requests

Use request.override() to create a modified request. This <b>follows an immutable pattern, leaving the original request unchanged.</b>

In [ ]:
async def double_args_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Double all numeric arguments before execution."""
    modified_args = {k: v * 2 for k, v in request.args.items()}
    modified_request = request.override(args=modified_args)  
    return await handler(modified_request)

# Original call: add(a=2, b=3) becomes add(a=4, b=6)

### Modifying headers at runtime

Interceptors can modify HTTP headers dynamically based on the request context:

In [ ]:
async def auth_header_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Add authentication headers based on the tool being called."""
    token = get_token_for_tool(request.name)
    modified_request = request.override(
        headers={"Authorization": f"Bearer {token}"}  
    )
    return await handler(modified_request)

## Composing interceptors

Multiple interceptors compose in “onion” order — the first interceptor in the list is the outermost layer:

In [ ]:
async def outer_interceptor(request, handler):
    print("outer: before")
    result = await handler(request)
    print("outer: after")
    return result

async def inner_interceptor(request, handler):
    print("inner: before")
    result = await handler(request)
    print("inner: after")
    return result

client = MultiServerMCPClient(
    {...},
    tool_interceptors=[outer_interceptor, inner_interceptor],  
)

# Execution order:
# outer: before -> inner: before -> tool execution -> inner: after -> outer: after

### Error handling

Use interceptors to catch tool execution errors and implement retry logic:

In [ ]:
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

client = MultiServerMCPClient(
    {...},
    tool_interceptors=[retry_interceptor],  
)

You can also catch specific error types and return fallback values:

In [ ]:
async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."

### Progress notifications

Subscribe to progress updates for long-running tool executions:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.callbacks import Callbacks, CallbackContext

async def on_progress(
    progress: float,
    total: float | None,
    message: str | None,
    context: CallbackContext,
):
    """Handle progress updates from MCP servers."""
    percent = (progress / total * 100) if total else progress
    tool_info = f" ({context.tool_name})" if context.tool_name else ""
    print(f"[{context.server_name}{tool_info}] Progress: {percent:.1f}% - {message}")

client = MultiServerMCPClient(
    {...},
    callbacks=Callbacks(on_progress=on_progress),  
)

The CallbackContext provides:

- server_name: Name of the MCP server
- tool_name: Name of the tool being executed (available during tool calls)


### Logging

The MCP protocol supports logging notifications from servers. Use the Callbacks class to subscribe to these events.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.callbacks import Callbacks, CallbackContext
from mcp.types import LoggingMessageNotificationParams

async def on_logging_message(
    params: LoggingMessageNotificationParams,
    context: CallbackContext,
):
    """Handle log messages from MCP servers."""
    print(f"[{context.server_name}] {params.level}: {params.data}")

client = MultiServerMCPClient(
    {...},
    callbacks=Callbacks(on_logging_message=on_logging_message),  
)

### Elicitation
<b>Elicitation</b> allows MCP servers to request additional input from users during tool execution. Instead of requiring all inputs upfront, servers can interactively ask for information as needed.
​
### Server setup
Define a tool that uses ctx.elicit() to request user input with a schema:

In [ ]:
from pydantic import BaseModel
from mcp.server.fastmcp import Context, FastMCP

server = FastMCP("Profile")

class UserDetails(BaseModel):
    email: str
    age: int

@server.tool()
async def create_profile(name: str, ctx: Context) -> str:
    """Create a user profile, requesting details via elicitation."""
    result = await ctx.elicit(  
        message=f"Please provide details for {name}'s profile:",  
        schema=UserDetails,  
    )  
    if result.action == "accept" and result.data:
        return f"Created profile for {name}: email={result.data.email}, age={result.data.age}"
    if result.action == "decline":
        return f"User declined. Created minimal profile for {name}."
    return "Profile creation cancelled."

if __name__ == "__main__":
    server.run(transport="http")


### Client setup

Handle elicitation requests by providing a callback to MultiServerMCPClient:

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.callbacks import Callbacks, CallbackContext
from mcp.shared.context import RequestContext
from mcp.types import ElicitRequestParams, ElicitResult

async def on_elicitation(
    mcp_context: RequestContext,
    params: ElicitRequestParams,
    context: CallbackContext,
) -> ElicitResult:
    """Handle elicitation requests from MCP servers."""
    # In a real application, you would prompt the user for input
    # based on params.message and params.requestedSchema
    return ElicitResult(  
        action="accept",  
        content={"email": "user@example.com", "age": 25},  
    )  

client = MultiServerMCPClient(
    {
        "profile": {
            "url": "http://localhost:8000/mcp",
            "transport": "http",
        }
    },
    callbacks=Callbacks(on_elicitation=on_elicitation),  
)

### Response actions

The elicitation callback can return one of three actions:

<img src='elicitation_actions.jpg'>

In [ ]:
# Accept with data
ElicitResult(action="accept", content={"email": "user@example.com", "age": 25})

# Decline (user doesn't want to provide info)
ElicitResult(action="decline")

# Cancel (abort the operation)
ElicitResult(action="cancel")